In [1]:
from google.colab import files
uploaded = files.upload()

Saving departments.csv to departments.csv
Saving employees_nested.json to employees_nested.json
Saving employees.csv to employees.csv
Saving sales.csv to sales.csv


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Datastructure examples').getOrCreate()

print("Spark session created...!")

Spark session created...!


In [13]:
employees_df=spark.read.csv("employees.csv",header=True,inferSchema=True)
employees_df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [12]:
departments_df=spark.read.csv("departments.csv",header=True,inferSchema=True)
departments_df.show()

+----------+---------+
|department| location|
+----------+---------+
|        IT|Bangalore|
|        HR|   Mumbai|
|   Finance|    Delhi|
+----------+---------+



In [11]:
sales_df=spark.read.csv("sales.csv",header=True,inferSchema=True)
sales_df.show()

+-------+------+--------+------+
|sale_id|emp_id| product|amount|
+-------+------+--------+------+
|      1|     1|  Laptop| 75000|
|      2|     2|   Mouse|  1500|
|      3|     3|Keyboard|  3000|
|      4|     1| Monitor| 12000|
|      5|     4|  Laptop| 75000|
|      6|     3|   Mouse|  1000|
|      7|     5|Keyboard|  1500|
|      8|     1|  Laptop| 75000|
+-------+------+--------+------+



In [10]:
emp_nested_df = spark.read.option("multiline", "true").json("employees_nested.json")
emp_nested_df.show(truncate=False)

+----------------------+------+-----+------------------------+
|address               |emp_id|name |skills                  |
+----------------------+------+-----+------------------------+
|{Hyderabad, Telangana}|1     |Rahul|[Python, SQL]           |
|{Bangalore, Karnataka}|2     |Sneha|[HR, Recruitment]       |
|{Chennai, Tamil Nadu} |3     |Arjun|[PySpark, Azure, Python]|
+----------------------+------+-----+------------------------+



###**PART 12 — When / Otherwise**
Create salary category:    
\>= 75000 → High     
60000–74999 → Medium   
< 60000 → Low

**Exercise 39**

Create new column salary_grade .

In [52]:
from pyspark.sql.functions import when

employees_df = employees_df.withColumn(
    "salary_grade",
    when(col("salary") >= 75000, "High")
    .when(col("salary") >= 60000, "Medium")
    .otherwise("Low")
).show()

+------+-----+----------+------+------------+------+---------+------------+
|emp_id| name|department|salary|joining_date| bonus|  company|salary_grade|
+------+-----+----------+------+------------+------+---------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|BotCampus|      Medium|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|BotCampus|      Medium|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|BotCampus|        High|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|BotCampus|        High|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|BotCampus|         Low|
|     6|Meera|      NULL| 72000|  2023-04-10|7200.0|BotCampus|      Medium|
|     7| Amit|        HR| 58000|  2023-01-18|5800.0|BotCampus|         Low|
+------+-----+----------+------+------------+------+---------+------------+



###**PART 13 — Date Functions**

**Exercise 40**

Convert joining_date to proper date format.

In [58]:
employees_df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

from pyspark.sql.functions import to_date, col

employees_df = employees_df.withColumn(
    "joining_date",
    to_date(col("joining_date"), "yyyy-MM-dd")
)
employees_df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



**Exercise 41**

Extract:
joining year,
joining month

In [60]:
from pyspark.sql.functions import year, month

employees_df = employees_df \
    .withColumn("joining_year", year(col("joining_date"))) \
    .withColumn("joining_month", month(col("joining_date")))
employees_df.show()

+------+-----+----------+------+------------+------------+-------------+
|emp_id| name|department|salary|joining_date|joining_year|joining_month|
+------+-----+----------+------+------------+------------+-------------+
|     1|Rahul|        IT| 70000|  2023-01-10|        2023|            1|
|     2|Sneha|        HR| 60000|  2022-11-15|        2022|           11|
|     3|Arjun|        IT| 75000|  2023-03-01|        2023|            3|
|     4|Priya|   Finance| 80000|  2022-12-20|        2022|           12|
|     5|Karan|        IT| 50000|  2023-02-05|        2023|            2|
|     6|Meera|      NULL| 72000|  2023-04-10|        2023|            4|
|     7| Amit|        HR| 58000|  2023-01-18|        2023|            1|
+------+-----+----------+------+------------+------------+-------------+



###**PART 14 — Union**
Create another DataFrame with 2 new employees.

In [64]:
new_data = [
    (8, "Riya", "Finance", 67000, "2023-05-10"),
    (9, "Vikram", "IT", 72000, "2023-06-01")
]
columns = ["emp_id", "name", "department", "salary", "joining_date"]
new_df = spark.createDataFrame(new_data, columns)
new_df.show()

+------+------+----------+------+------------+
|emp_id|  name|department|salary|joining_date|
+------+------+----------+------+------------+
|     8|  Riya|   Finance| 67000|  2023-05-10|
|     9|Vikram|        IT| 72000|  2023-06-01|
+------+------+----------+------+------------+



**Exercise** **42**

Union the two DataFrames.

In [65]:
union_df = employees_df.union(new_df)
union_df.show()

AnalysisException: [NUM_COLUMNS_MISMATCH] UNION can only be performed on inputs with the same number of columns, but the first input has 7 columns and the second input has 5 columns. SQLSTATE: 42826;
'Union false, false
:- Project [emp_id#945, name#946, department#947, salary#948, joining_date#951, joining_year#973, month(joining_date#951) AS joining_month#974]
:  +- Project [emp_id#945, name#946, department#947, salary#948, joining_date#951, year(joining_date#951) AS joining_year#973]
:     +- Project [emp_id#945, name#946, department#947, salary#948, to_date(joining_date#949, Some(yyyy-MM-dd), Some(Etc/UTC), true) AS joining_date#951]
:        +- Relation [emp_id#945,name#946,department#947,salary#948,joining_date#949] csv
+- LogicalRDD [emp_id#1044L, name#1045, department#1046, salary#1047L, joining_date#1048], false


**Exercise 43**

Use unionByName .

In [66]:
union_df = employees_df.unionByName(new_df, allowMissingColumns=True)
union_df.show()

+------+------+----------+------+------------+------------+-------------+
|emp_id|  name|department|salary|joining_date|joining_year|joining_month|
+------+------+----------+------+------------+------------+-------------+
|     1| Rahul|        IT| 70000|  2023-01-10|        2023|            1|
|     2| Sneha|        HR| 60000|  2022-11-15|        2022|           11|
|     3| Arjun|        IT| 75000|  2023-03-01|        2023|            3|
|     4| Priya|   Finance| 80000|  2022-12-20|        2022|           12|
|     5| Karan|        IT| 50000|  2023-02-05|        2023|            2|
|     6| Meera|      NULL| 72000|  2023-04-10|        2023|            4|
|     7|  Amit|        HR| 58000|  2023-01-18|        2023|            1|
|     8|  Riya|   Finance| 67000|  2023-05-10|        NULL|         NULL|
|     9|Vikram|        IT| 72000|  2023-06-01|        NULL|         NULL|
+------+------+----------+------+------------+------------+-------------+



###**PART 15 — Window Functions**

**Exercise 44**

Rank employees by salary within department.

In [67]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, col
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

employees_df = employees_df.withColumn(
    "salary_rank",
    rank().over(window_spec)
)
employees_df.show()

+------+-----+----------+------+------------+------------+-------------+-----------+
|emp_id| name|department|salary|joining_date|joining_year|joining_month|salary_rank|
+------+-----+----------+------+------------+------------+-------------+-----------+
|     6|Meera|      NULL| 72000|  2023-04-10|        2023|            4|          1|
|     4|Priya|   Finance| 80000|  2022-12-20|        2022|           12|          1|
|     2|Sneha|        HR| 60000|  2022-11-15|        2022|           11|          1|
|     7| Amit|        HR| 58000|  2023-01-18|        2023|            1|          2|
|     3|Arjun|        IT| 75000|  2023-03-01|        2023|            3|          1|
|     1|Rahul|        IT| 70000|  2023-01-10|        2023|            1|          2|
|     5|Karan|        IT| 50000|  2023-02-05|        2023|            2|          3|
+------+-----+----------+------+------------+------------+-------------+-----------+



**Exercise 45**

Assign row number within department.

In [68]:
employees_df = employees_df.withColumn(
    "row_number",
    row_number().over(window_spec)
)
employees_df.show()

+------+-----+----------+------+------------+------------+-------------+-----------+----------+
|emp_id| name|department|salary|joining_date|joining_year|joining_month|salary_rank|row_number|
+------+-----+----------+------+------------+------------+-------------+-----------+----------+
|     6|Meera|      NULL| 72000|  2023-04-10|        2023|            4|          1|         1|
|     4|Priya|   Finance| 80000|  2022-12-20|        2022|           12|          1|         1|
|     2|Sneha|        HR| 60000|  2022-11-15|        2022|           11|          1|         1|
|     7| Amit|        HR| 58000|  2023-01-18|        2023|            1|          2|         2|
|     3|Arjun|        IT| 75000|  2023-03-01|        2023|            3|          1|         1|
|     1|Rahul|        IT| 70000|  2023-01-10|        2023|            1|          2|         2|
|     5|Karan|        IT| 50000|  2023-02-05|        2023|            2|          3|         3|
+------+-----+----------+------+--------

###**PART 16 — Working with Nested JSON**

**Exercise 46**

Read employees_nested.json .

In [69]:
from pyspark.sql import SparkSession
nested_df = spark.read.option("multiline", "true").json("employees_nested.json")
nested_df.show(truncate=False)
nested_df.printSchema()

+----------------------+------+-----+------------------------+
|address               |emp_id|name |skills                  |
+----------------------+------+-----+------------------------+
|{Hyderabad, Telangana}|1     |Rahul|[Python, SQL]           |
|{Bangalore, Karnataka}|2     |Sneha|[HR, Recruitment]       |
|{Chennai, Tamil Nadu} |3     |Arjun|[PySpark, Azure, Python]|
+----------------------+------+-----+------------------------+

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



**Exercise 47**

Extract:
emp_id
city
state

In [70]:
nested_selected_df = nested_df.select(
    col("emp_id"),
    col("address.city").alias("city"),
    col("address.state").alias("state")
)
nested_selected_df.show()

+------+---------+----------+
|emp_id|     city|     state|
+------+---------+----------+
|     1|Hyderabad| Telangana|
|     2|Bangalore| Karnataka|
|     3|  Chennai|Tamil Nadu|
+------+---------+----------+



**Exercise 48**

Explode the skills array.   
Expected output example:   
Rahul → Python  
Rahul → SQL

In [71]:
from pyspark.sql.functions import explode

exploded_df = nested_df.select(
    col("name"),
    explode(col("skills")).alias("skill")
)
exploded_df.show()

+-----+-----------+
| name|      skill|
+-----+-----------+
|Rahul|     Python|
|Rahul|        SQL|
|Sneha|         HR|
|Sneha|Recruitment|
|Arjun|    PySpark|
|Arjun|      Azure|
|Arjun|     Python|
+-----+-----------+



###**PART 17 — Sales Analysis**
Using sales.csv

**Exercise 49**

Calculate total sales.

In [72]:
from pyspark.sql.functions import sum

sales_df.select(
    sum("amount").alias("total_sales")
).show()

+-----------+
|total_sales|
+-----------+
|     244000|
+-----------+



**Exercise 50**

Find total sales per employee.

In [73]:
sales_per_emp = sales_df.groupBy("emp_id") \
    .agg(sum("amount").alias("total_sales"))
sales_per_emp.show()

+------+-----------+
|emp_id|total_sales|
+------+-----------+
|     1|     162000|
|     3|       4000|
|     5|       1500|
|     4|      75000|
|     2|       1500|
+------+-----------+



**Exercise 51**

Find best performing employee.

In [74]:
sales_per_emp.orderBy(col("total_sales").desc()).show(1)

+------+-----------+
|emp_id|total_sales|
+------+-----------+
|     1|     162000|
+------+-----------+
only showing top 1 row


**Exercise 52**

Find total sales per product.

In [75]:
sales_df.groupBy("product") \
    .agg(sum("amount").alias("total_sales")) \
    .show()

+--------+-----------+
| product|total_sales|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       2500|
|Keyboard|       4500|
| Monitor|      12000|
+--------+-----------+



###**PART 18 — Applying Functions**
Create a custom function that categorizes sales:   
\> 50000 → Premium    
20000–50000 → Medium  
< 20000 → Small

**Exercise 53**

Apply this function to create sale_category .

In [76]:
from pyspark.sql.functions import when, col

sales_df = sales_df.withColumn(
    "sale_category",
    when(col("amount") > 50000, "Premium")
    .when((col("amount") >= 20000) & (col("amount") <= 50000), "Medium")
    .otherwise("Small")
)
sales_df.show()

+-------+------+--------+------+-------------+
|sale_id|emp_id| product|amount|sale_category|
+-------+------+--------+------+-------------+
|      1|     1|  Laptop| 75000|      Premium|
|      2|     2|   Mouse|  1500|        Small|
|      3|     3|Keyboard|  3000|        Small|
|      4|     1| Monitor| 12000|        Small|
|      5|     4|  Laptop| 75000|      Premium|
|      6|     3|   Mouse|  1000|        Small|
|      7|     5|Keyboard|  1500|        Small|
|      8|     1|  Laptop| 75000|      Premium|
+-------+------+--------+------+-------------+



###**PART 19 — Pandas Comparison**

**Exercise 54**

Convert Spark DataFrame to Pandas.

In [77]:
pandas_df = employees_df.toPandas()
print(pandas_df.head())

   emp_id   name department  salary joining_date  joining_year  joining_month  \
0       6  Meera       None   72000   2023-04-10          2023              4   
1       4  Priya    Finance   80000   2022-12-20          2022             12   
2       2  Sneha         HR   60000   2022-11-15          2022             11   
3       7   Amit         HR   58000   2023-01-18          2023              1   
4       3  Arjun         IT   75000   2023-03-01          2023              3   

   salary_rank  row_number  
0            1           1  
1            1           1  
2            1           1  
3            2           2  
4            1           1  


**Exercise 55**

Perform salary filtering using Pandas.

In [78]:
filtered_df = pandas_df[pandas_df["salary"] > 50000]
print(filtered_df)

   emp_id   name department  salary joining_date  joining_year  joining_month  \
0       6  Meera       None   72000   2023-04-10          2023              4   
1       4  Priya    Finance   80000   2022-12-20          2022             12   
2       2  Sneha         HR   60000   2022-11-15          2022             11   
3       7   Amit         HR   58000   2023-01-18          2023              1   
4       3  Arjun         IT   75000   2023-03-01          2023              3   
5       1  Rahul         IT   70000   2023-01-10          2023              1   

   salary_rank  row_number  
0            1           1  
1            1           1  
2            1           1  
3            2           2  
4            1           1  
5            2           2  


###**PART 20 — Widgets in Colab**
Create widgets to filter employees.

**Exercise 56**

Create dropdown for department.

In [79]:
import ipywidgets as widgets
from IPython.display import display

departments = [row["department"] for row in employees_df.select("department").distinct().collect() if row["department"] is not None]
dept_dropdown = widgets.Dropdown(
    options=departments,
    description="Department:",
    value=departments[0]
)
display(dept_dropdown)

Dropdown(description='Department:', options=('HR', 'Finance', 'IT'), value='HR')

**Exercise 57**

Create slider for salary.

In [80]:
salary_slider = widgets.IntSlider(
    value=50000,
    min=0,
    max=100000,
    step=5000,
    description="Min Salary:"
)
display(salary_slider)

IntSlider(value=50000, description='Min Salary:', max=100000, step=5000)

**Exercise 58**

Display employees based on widget values.

In [81]:
from pyspark.sql.functions import col

def filter_employees(department, min_salary):
    filtered_df = employees_df.filter(
        (col("department") == department) &
        (col("salary") >= min_salary)
    )
    filtered_df.show()
widgets.interactive(
    filter_employees,
    department=dept_dropdown,
    min_salary=salary_slider
)

interactive(children=(Dropdown(description='Department:', index=2, options=('HR', 'Finance', 'IT'), value='IT'…

###**Final Mini Project**

Using all datasets, generate report:
Top Employee,
Average Salary,
Department Employee Count,
Total Sales,
Best Sales Employee  
Save results to:
final_report.txt

In [82]:
from pyspark.sql.functions import avg, sum, col
top_employee = employees_df.orderBy(col("salary").desc()).select("name").first()[0]
avg_salary = employees_df.select(avg("salary")).first()[0]
dept_count_df = employees_df.groupBy("department").count()
dept_counts = dept_count_df.collect()
total_sales = sales_df.select(sum("amount")).first()[0]
sales_per_emp = sales_df.groupBy("emp_id") \
    .agg(sum("amount").alias("total_sales"))
best_emp_df = sales_per_emp.orderBy(col("total_sales").desc()) \
    .join(employees_df, "emp_id")
best_sales_employee = best_emp_df.select("name").first()[0]
report = f"""
===== FINAL REPORT =====
Top Employee (Highest Salary): {top_employee}
Average Salary: {avg_salary}
Department Employee Count:
"""
for row in dept_counts:
    report += f"{row['department']} : {row['count']}\n"
report += f"""
Total Sales: {total_sales}
Best Sales Employee: {best_sales_employee}
"""
with open("final_report.txt", "w") as file:
    file.write(report)

print("Report saved successfully!")

Report saved successfully!


In [83]:
with open("final_report.txt", "r") as file:
    print(file.read())


===== FINAL REPORT =====
Top Employee (Highest Salary): Priya
Average Salary: 66428.57142857143
Department Employee Count:
HR : 2
None : 1
Finance : 1
IT : 3

Total Sales: 244000
Best Sales Employee: Rahul

